# 21 — O1-S exact interval scan (GPU, DEV-only)

Notebook này chạy sau notebook 20. Notebook 20 đã cho thấy LB64 có tín hiệu nhưng
61/71 DEV query-event vẫn có interval dài hơn 64 frame, vì vậy notebook 21 quét
đúng **từng frame nguyên trong inclusive GT interval** của 61 event đó. Đây là
feature scan chẩn đoán để đo best source thật, không phải selector production.

Hai runner `quang` và `trong` dùng cùng một plan bất biến; mỗi video chỉ thuộc một
runner. Mỗi video tạo deterministic JPEG tar, sample manifest, SigLIP matrix và
report. Chỉ sau khi upload + đọc lại hash thành công mới ghi `COMPLETE` marker.
Rerun đúng runner sẽ bỏ qua video đã có marker hợp lệ, nên không cần chạy lại từ đầu.

### Cách chạy

1. Import notebook từ nhánh `feature/quang/storage`, chọn GPU, bật Internet.
2. Bật Kaggle Secrets `GITHUB_TOKEN`, `MODAL_TOKEN_ID`, `MODAL_TOKEN_SECRET`.
3. Quang đặt `RUNNER="quang"`; Trọng đặt `RUNNER="trong"`.
4. Chạy kiểm tra nhanh: giữ `RUN_SCOPE="PREFLIGHT"`, bấm Run All. Bước này chỉ
   tải repo/model, xác minh lineage và numeric canary; không quét video.
5. Chạy thật: đổi `RUN_SCOPE="SHARD"`, đặt `CONFIRM_SHARD` đúng bằng `RUNNER`,
   rồi Save Version. Nếu bị ngắt, chạy lại cùng cấu hình để resume.
6. Chỉ khi cả hai manifest đều `COMPLETE` mới chạy notebook 22 aggregate.

Notebook chỉ đọc BTC và notebook 18/20 artifacts, ghi vào volume research; không
đọc held-out, không mutate `keyframes-btc`, không tự kết luận official score uplift.
Full feature parity OCR/caption/BGE/downstream vẫn là gate sau đó.

## 0 — Chỉ sửa RUNNER / RUN_SCOPE

In [ ]:
from pathlib import Path

RUNNER = "quang"                    # chỉ "quang" hoặc "trong"
RUNNERS = ("quang", "trong")
RUN_SCOPE = "PREFLIGHT"             # PREFLIGHT hoặc SHARD
CONFIRM_SHARD = ""                  # SHARD yêu cầu đúng RUNNER
SESSION_MAX_MINUTES = 600
SIGLIP_BATCH_SIZE = 32
JPEG_QUALITY = 95
CANARY_TOLERANCE = 1e-5

RUN_ID = "o1s-exact-v1"
INTERVAL_POLICY = "inclusive_full_interval_exact"
EXPECTED_FULL_TARGETS = 61
EXPECTED_NB18_REPORT_SHA256 = "b089835a679809cfb1d2a1f79580343a8ceb0fe34846452038101f0db3a59228"
EXPECTED_NB18_REPORT_PAYLOAD_HASH = "96f5169042c621b3ee654b5f356e21240aef3af9d708e80ecc9a79d065dbd93c"
EXPECTED_TARGETS_SHA256 = "6929818a00205179f1d96c394f2930f43f88e9d536edccaf169b0632b5ee716b"
EXPECTED_AGGREGATE_COMPLETE_SHA256 = "27349b5a3b2aa0e2858934106808224bef893cf938e01616c0bb8d9e647b4804"
REFERENCE_LB64_CANARY_REPORT_HASH = "45828774efcb0e291538b3405e41c8c720c9bb4e5c4e6588401c7a0547ca5a3c"

REPO_URL = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF = "feature/quang/storage"
REPO_DIR = Path("/kaggle/working/multimodal-fusion-engine")
WORK_DIR = Path(f"/kaggle/working/{RUN_ID}-{RUNNER}")

RESEARCH_VOLUME = "keyframe-augmentation-research"
BTC_VOLUME = "keyframes-btc"
BTC_BUNDLE = "aic_btc_b1_v1"
UPSTREAM_ROOT = "/research/keyframe-augmentation/transnetv2-btc-b1-v1"
NB18_ROOT = f"{UPSTREAM_ROOT}/gate1-oracle-eval-quang-v1"
NB20_ROOT = f"{UPSTREAM_ROOT}/gate1-o1s-lb64-aggregate-v1"
LB64_ROOT = f"{UPSTREAM_ROOT}/gate1-o1s-lb64-scan-v1"
OUTPUT_ROOT = f"{UPSTREAM_ROOT}/gate1-o1s-exact-v1"

CHECKPOINT = "google/siglip-so400m-patch14-384"
MODEL_REVISION = "9fdffc58afc957d1a03a25b10dba0329ab15c2a3"
SIGLIP_DIMENSION = 1152
CANARY_VIDEO_ID = "L21_V001"

if RUN_SCOPE not in {"PREFLIGHT", "SHARD"}:
    raise ValueError("RUN_SCOPE phải là PREFLIGHT hoặc SHARD")
if RUNNER not in RUNNERS:
    raise ValueError(f"RUNNER phải thuộc {RUNNERS}")
if RUN_SCOPE == "SHARD" and CONFIRM_SHARD != RUNNER:
    raise RuntimeError("SHARD bị khóa; đặt CONFIRM_SHARD đúng bằng RUNNER")
if not 1 <= SIGLIP_BATCH_SIZE <= 64:
    raise ValueError("SIGLIP_BATCH_SIZE phải trong [1, 64]")
WORK_DIR.mkdir(parents=True, exist_ok=True)

## 1 — Refresh repo, lineage notebook 18/20 và tạo exact plan

In [ ]:
import csv
import gc
import json
import os
import shutil
import subprocess
import sys
import time
from collections import defaultdict


def run(command: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None) -> str:
    result = subprocess.run(command, cwd=cwd, env=env, check=False, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(f"Command failed: {' '.join(command[:3])}\n{result.stderr[-1500:]}")
    return result.stdout.strip()


def refresh_repo() -> None:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    askpass = WORK_DIR / "git-askpass.sh"
    askpass.write_text(
        '#!/bin/sh\ncase "$1" in *Username*) printf x-access-token ;; *) printf %s "$GITHUB_TOKEN" ;; esac\n',
        encoding="utf-8",
    )
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update({"GITHUB_TOKEN": token, "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0"})
    try:
        if (REPO_DIR / ".git").is_dir():
            if run(["git", "status", "--porcelain"], cwd=REPO_DIR):
                raise RuntimeError("Checkout Kaggle có local changes; Restart Session rồi Run All")
            run(["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=REPO_DIR, env=env)
            run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR)
        else:
            run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], env=env)
    finally:
        askpass.unlink(missing_ok=True)


refresh_repo()
requirements = REPO_DIR / "notebooks/requirements-kaggle-btc-images.txt"


def torch_gpu_probe() -> dict:
    payload = run([
        sys.executable,
        "-c",
        "import json,torch; c=torch.cuda.get_device_capability(0); "
        "print(json.dumps({'torch':torch.__version__,'gpu':torch.cuda.get_device_name(0),"
        "'capability':list(c),'arch_list':torch.cuda.get_arch_list()}))",
    ])
    return json.loads(payload.splitlines()[-1])



TORCH_PROBE = torch_gpu_probe()
required_arch = f"sm_{TORCH_PROBE['capability'][0]}{TORCH_PROBE['capability'][1]}"
if required_arch not in TORCH_PROBE["arch_list"]:
    raise RuntimeError(
        "GPU không tương thích với PyTorch hiện tại: "
        f"{TORCH_PROBE}. Chọn T4/L4/A10; không chọn P100 cho notebook SigLIP này."
    )
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)])
run([sys.executable, "-m", "pip", "install", "-q", "modal", "av"])
sys.path.insert(0, str(REPO_DIR / "src"))

import av
import numpy as np
import torch
from mfe.contracts.ids import build_keyframe_id
from mfe.offline.augmentation.io import canonical_hash, read_jsonl, sha256_file, write_json_atomic, write_jsonl_atomic
from mfe.offline.augmentation.oracle_artifacts import OracleArtifactTarget
from mfe.offline.augmentation.representative_oracle import (
    RepresentativeResolvedSample,
    RepresentativeScanRequest,
    build_representative_scan_plan,
    partition_representative_scan,
    representative_scan_plan_hash,
    validate_scan_completion_marker,
)
from mfe.offline.bundle import BundleLayout, check_vectors, freeze_environment
from mfe.offline.keyframe_augmentation.common import sha256_json
from mfe.offline.keyframe_augmentation.delta_siglip import validate_siglip_baseline_manifest, write_deterministic_image_tar
from mfe.offline.source import fetch_video, load_catalogue
from mfe.offline.stages import _siglip_feature_tensor
from mfe.offline.transport import ModalVolumeTransport

if not torch.cuda.is_available():
    raise RuntimeError("Không thấy CUDA; chọn GPU accelerator rồi Restart Session")

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["MODAL_TOKEN_ID"] = secrets.get_secret("MODAL_TOKEN_ID")
    os.environ["MODAL_TOKEN_SECRET"] = secrets.get_secret("MODAL_TOKEN_SECRET")
except Exception as error:
    raise RuntimeError("Thiếu/bị tắt Modal secrets") from error

RESEARCH = ModalVolumeTransport(RESEARCH_VOLUME, create_if_missing=False, version=2)
BTC = ModalVolumeTransport(BTC_VOLUME, create_if_missing=False, version=2)
BTC_LAYOUT = BundleLayout(BTC_BUNDLE)
CATALOGUE = load_catalogue(REPO_DIR / "config/zips_batch1.yaml")
REPO_COMMIT = run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR)

INPUT_DIR = WORK_DIR / "input"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT18_PATH = INPUT_DIR / "oracle-eval-report.json"
TARGETS_PATH = INPUT_DIR / "oracle-targets-resolved.jsonl"
RESEARCH.get(f"{NB18_ROOT}/oracle-eval-report.json", REPORT18_PATH)
RESEARCH.get(f"{NB18_ROOT}/oracle-targets-resolved.jsonl", TARGETS_PATH)
if sha256_file(REPORT18_PATH) != EXPECTED_NB18_REPORT_SHA256:
    raise RuntimeError("Notebook 18 report SHA drift")
if sha256_file(TARGETS_PATH) != EXPECTED_TARGETS_SHA256:
    raise RuntimeError("Notebook 18 target SHA drift")
report18 = json.loads(REPORT18_PATH.read_text(encoding="utf-8"))
unsigned18 = dict(report18)
claimed18 = unsigned18.pop("report_payload_hash", None)
if claimed18 != EXPECTED_NB18_REPORT_PAYLOAD_HASH or canonical_hash(unsigned18) != claimed18:
    raise RuntimeError("Notebook 18 report payload hash invalid")
if report18.get("status") != "ORACLE_DIAGNOSTIC_COMPLETE" or report18.get("heldout_accessed") is not False:
    raise RuntimeError("Notebook 18 lineage/split invalid")
if report18.get("btc_mutated") is not False:
    raise RuntimeError("Notebook 18 does not prove BTC immutability")


NB20_COMPLETE_PATH = INPUT_DIR / "nb20-COMPLETE.json"
RESEARCH.get(f"{NB20_ROOT}/COMPLETE.json", NB20_COMPLETE_PATH)
if sha256_file(NB20_COMPLETE_PATH) != EXPECTED_AGGREGATE_COMPLETE_SHA256:
    raise RuntimeError("Notebook 20 COMPLETE marker SHA drift")
nb20_complete = json.loads(NB20_COMPLETE_PATH.read_text(encoding="utf-8"))
if (
    nb20_complete.get("status") != "COMPLETE"
    or nb20_complete.get("heldout_accessed") is not False
    or nb20_complete.get("btc_mutated") is not False
):
    raise RuntimeError("Notebook 20 lineage/status is not a verified diagnostic COMPLETE")

TARGETS = [OracleArtifactTarget.model_validate(row) for row in read_jsonl(TARGETS_PATH)]
if len(TARGETS) != 219:
    raise RuntimeError(f"Expected 219 oracle targets, got {len(TARGETS)}")

FULL_TARGETS = [
    target
    for target in TARGETS
    if target.frame_range_end - target.frame_range_start + 1 > 64
]
if len(FULL_TARGETS) != EXPECTED_FULL_TARGETS:
    raise RuntimeError(
        f"Exact interval target count drift: {len(FULL_TARGETS)} != {EXPECTED_FULL_TARGETS}"
    )

by_frame = defaultdict(set)
for target in FULL_TARGETS:
    membership = (
        target.query_id,
        target.event_index,
        target.frame_range_start,
        target.frame_range_end,
    )
    for frame_idx in range(target.frame_range_start, target.frame_range_end + 1):
        by_frame[(target.video_id, frame_idx)].add(membership)

FULL_PLAN = []
for (video_id, frame_idx), memberships in sorted(by_frame.items()):
    FULL_PLAN.append(
        {
            "keyframe_id": build_keyframe_id(video_id, frame_idx),
            "video_id": video_id,
            "frame_idx": frame_idx,
            "interval_policy": INTERVAL_POLICY,
            "memberships": [
                {
                    "query_id": query_id,
                    "event_index": event_index,
                    "frame_range_start": start,
                    "frame_range_end": end,
                }
                for query_id, event_index, start, end in sorted(memberships)
            ],
        }
    )
FULL_PLAN_HASH = canonical_hash(FULL_PLAN)
if not FULL_PLAN:
    raise RuntimeError("Exact interval plan is empty")

video_costs = {video_id: entry.size for video_id, entry in CATALOGUE.items()}
by_video = defaultdict(list)
for row in FULL_PLAN:
    by_video[row["video_id"]].append(row)
totals = {runner: 0 for runner in RUNNERS}
PARTITIONS = {runner: [] for runner in RUNNERS}
for video_id in sorted(by_video, key=lambda value: (-video_costs[value], value)):
    owner = min(RUNNERS, key=lambda value: (totals[value], value))
    PARTITIONS[owner].extend(by_video[video_id])
    totals[owner] += video_costs[video_id]
for owner in PARTITIONS:
    PARTITIONS[owner].sort(key=lambda row: (row["video_id"], row["frame_idx"]))

ASSIGNED = PARTITIONS[RUNNER]
ASSIGNED_VIDEO_IDS = sorted({row["video_id"] for row in ASSIGNED})
BY_VIDEO = defaultdict(list)
for row in ASSIGNED:
    BY_VIDEO[row["video_id"]].append(row)
full_keys = {(row["video_id"], row["frame_idx"]) for row in FULL_PLAN}
partition_keys = {
    (row["video_id"], row["frame_idx"])
    for owner in RUNNERS
    for row in PARTITIONS[owner]
}
if partition_keys != full_keys:
    raise RuntimeError("Runner partitions do not cover the exact plan")
quang_keys = {(row["video_id"], row["frame_idx"]) for row in PARTITIONS["quang"]}
trong_keys = {(row["video_id"], row["frame_idx"]) for row in PARTITIONS["trong"]}
if quang_keys.intersection(trong_keys):
    raise RuntimeError("Runner partitions overlap")
print({
    "repo_commit": REPO_COMMIT,
    "gpu": torch.cuda.get_device_name(0),
    "runner": RUNNER,
    "scope": RUN_SCOPE,
    "full_interval_targets": len(FULL_TARGETS),
    "exact_unique_frames": len(FULL_PLAN),
    "assigned_videos": len(ASSIGNED_VIDEO_IDS),
    "assigned_frames": len(ASSIGNED),
    "assigned_source_gib": round(sum(video_costs[v] for v in ASSIGNED_VIDEO_IDS) / 2**30, 3),
    "interval_policy": INTERVAL_POLICY,
    "full_plan_hash": FULL_PLAN_HASH,
    "decision_authority": "diagnostic_only",
})

## 2 — Numeric canary trên BTC JPEG + stored SigLIP vector

In [ ]:
from huggingface_hub import snapshot_download
from PIL import Image
from transformers import AutoModel, AutoProcessor

SNAPSHOT_DIR = Path(snapshot_download(CHECKPOINT, revision=MODEL_REVISION))
MODEL_FILE_HASHES = {
    str(path.relative_to(SNAPSHOT_DIR)): sha256_file(path)
    for path in sorted(SNAPSHOT_DIR.rglob("*"))
    if path.is_file()
    and (path.suffix in {".json", ".safetensors", ".txt"} or path.name.startswith("tokenizer"))
}
MODEL_SNAPSHOT_HASH = sha256_json(MODEL_FILE_HASHES)
PROCESSOR = AutoProcessor.from_pretrained(SNAPSHOT_DIR, use_fast=True)
MODEL = AutoModel.from_pretrained(SNAPSHOT_DIR).eval().cuda()

CANARY_DIR = WORK_DIR / "canary"
CANARY_DIR.mkdir(parents=True, exist_ok=True)
baseline_manifest_path = CANARY_DIR / "part-manifest.json"
baseline_vectors_path = CANARY_DIR / "siglip.npz"
baseline_mapping_path = CANARY_DIR / "mapping.csv"
BTC.get(BTC_LAYOUT.part_manifest(CANARY_VIDEO_ID), baseline_manifest_path)
BTC.get(BTC_LAYOUT.part("siglip_vectors", CANARY_VIDEO_ID), baseline_vectors_path)
BTC.get(BTC_LAYOUT.map_keyframes(CANARY_VIDEO_ID), baseline_mapping_path)
BASELINE_MANIFEST = json.loads(baseline_manifest_path.read_text(encoding="utf-8"))
BASELINE_VECTOR_SHA256 = sha256_file(baseline_vectors_path)
BASELINE_CONTRACT = validate_siglip_baseline_manifest(
    BASELINE_MANIFEST,
    vector_sha256=BASELINE_VECTOR_SHA256,
    checkpoint=CHECKPOINT,
    dimension=1152,
    batch_size=SIGLIP_BATCH_SIZE,
)
BASELINE_ENV = BASELINE_CONTRACT["stage_environment"]
CURRENT_ENV = freeze_environment()
ENV_FIELDS = ("torch", "transformers", "numpy")
ENV_MISMATCHES = {
    field: {"btc": BASELINE_ENV.get(field), "current": CURRENT_ENV.get(field)}
    for field in ENV_FIELDS
    if BASELINE_ENV.get(field) != CURRENT_ENV.get(field)
} if BASELINE_CONTRACT["stage_environment_status"] == "AVAILABLE" else {}
ENV_STATUS = (
    "MATCH" if BASELINE_CONTRACT["stage_environment_status"] == "AVAILABLE" and not ENV_MISMATCHES
    else "MISMATCH" if ENV_MISMATCHES
    else "UNVERIFIED_LEGACY_MANIFEST"
)

stored = np.load(baseline_vectors_path, allow_pickle=False)
stored_ids = stored["ids"].astype(str).tolist()
stored_by_id = {value: index for index, value in enumerate(stored_ids)}
with baseline_mapping_path.open("r", encoding="utf-8-sig", newline="") as stream:
    mapping_rows = list(csv.DictReader(stream))
canary_row = next(
    row
    for row in mapping_rows
    if build_keyframe_id(CANARY_VIDEO_ID, int(row["frame_idx"])) in stored_by_id
)
CANARY_ID = build_keyframe_id(CANARY_VIDEO_ID, int(canary_row["frame_idx"]))
canary_image_path = CANARY_DIR / "btc-canary.jpg"
BTC.get(f"{BTC_LAYOUT.keyframes_dir(CANARY_VIDEO_ID)}/{int(canary_row['n']):03d}.jpg", canary_image_path)

with Image.open(canary_image_path) as source:
    canary_image = source.convert("RGB")
inputs = PROCESSOR(images=[canary_image], return_tensors="pt").to("cuda")
with torch.inference_mode():
    raw = _siglip_feature_tensor(MODEL.get_image_features(**inputs)).float().cpu().numpy()
checked = check_vectors([CANARY_ID], raw, CANARY_VIDEO_ID, 1152)
current_vector = checked.kept[0]
baseline_vector = stored["vectors"][stored_by_id[CANARY_ID]].astype(np.float32)
CANARY_MAX_ABS_DIFF = float(np.max(np.abs(current_vector - baseline_vector)))
CANARY_COSINE = float(
    np.dot(current_vector, baseline_vector)
    / (np.linalg.norm(current_vector) * np.linalg.norm(baseline_vector))
)
CANARY_PASS = CANARY_MAX_ABS_DIFF <= CANARY_TOLERANCE
ARTIFACT_CONTRACT_PASS = BASELINE_CONTRACT["artifact_contract_status"] == "PASS"
ENV_PASS = ENV_STATUS != "MISMATCH"
PREFLIGHT_PASS = CANARY_PASS and ARTIFACT_CONTRACT_PASS and ENV_PASS
CANARY_REPORT = {
    "schema_version": 1,
    "status": (
        "PASS_NUMERIC_CANARY_WITH_LEGACY_PROVENANCE_GAP"
        if PREFLIGHT_PASS and ENV_STATUS == "UNVERIFIED_LEGACY_MANIFEST"
        else "PASS" if PREFLIGHT_PASS else "BLOCKED"
    ),
    "canary_id": CANARY_ID,
    "btc_jpeg_sha256": sha256_file(canary_image_path),
    "btc_siglip_part_sha256": BASELINE_VECTOR_SHA256,
    "max_abs_diff": CANARY_MAX_ABS_DIFF,
    "cosine": CANARY_COSINE,
    "tolerance": CANARY_TOLERANCE,
    "environment_mismatches": ENV_MISMATCHES,
    "baseline_stage_environment_status": ENV_STATUS,
    "baseline_artifact_contract": BASELINE_CONTRACT,
    "model": CHECKPOINT,
    "resolved_revision": MODEL_REVISION,
    "model_snapshot_hash": MODEL_SNAPSHOT_HASH,
    "processor": "SiglipImageProcessor(use_fast=True)",
    "o1s_feature_parity": "MEASURED_SIGLIP_NUMERIC_CANARY",
    "full_feature_parity_status": "BLOCKED_MISSING_OCR_CAPTION_BGE_AND_DOWNSTREAM",
}
CANARY_REPORT["report_hash"] = sha256_json(CANARY_REPORT)
write_json_atomic(CANARY_DIR / "siglip-canary-report.json", CANARY_REPORT)
print({"SIGLIP_PREFLIGHT": CANARY_REPORT})
if not PREFLIGHT_PASS:
    raise RuntimeError("SigLIP preflight BLOCKED; không chạy O1-S shard")

## 3 — Resume inventory; PREFLIGHT không decode

In [ ]:
LOCAL = WORK_DIR / "local"
VIDEOS = WORK_DIR / "videos"
LOCAL.mkdir(parents=True, exist_ok=True)
VIDEOS.mkdir(parents=True, exist_ok=True)


def remote_paths(video_id: str) -> dict[str, str]:
    return {
        "images": f"{OUTPUT_ROOT}/images/{video_id}.tar",
        "samples": f"{OUTPUT_ROOT}/samples/{video_id}.jsonl",
        "siglip": f"{OUTPUT_ROOT}/siglip/{video_id}.npz",
        "report": f"{OUTPUT_ROOT}/reports/{video_id}.json",
        "marker": f"{OUTPUT_ROOT}/markers/{RUNNER}/{video_id}.json",
    }


def valid_remote_completion(video_id: str) -> dict | None:
    remote = remote_paths(video_id)
    if not RESEARCH.exists(remote["marker"]):
        return None
    marker_path = LOCAL / f"marker-{video_id}.json"
    try:
        RESEARCH.get(remote["marker"], marker_path)
        marker = json.loads(marker_path.read_text(encoding="utf-8"))
        payload = dict(marker)
        claimed = payload.pop("marker_payload_hash", None)
        artifacts = payload.get("artifacts")
        if claimed != canonical_hash(payload) or not isinstance(artifacts, dict):
            return None
        expected = {
            "status": "COMPLETE",
            "run_id": RUN_ID,
            "runner": RUNNER,
            "video_id": video_id,
            "full_plan_hash": FULL_PLAN_HASH,
            "interval_policy": INTERVAL_POLICY,
            "heldout_accessed": False,
            "btc_mutated": False,
        }
        if any(payload.get(key) != value for key, value in expected.items()):
            return None
        if set(artifacts) != set(remote[key] for key in ("images", "samples", "siglip", "report")):
            return None
        observed = {path: RESEARCH.sha256(path) for path in artifacts}
        return marker if all(observed.get(path) == digest for path, digest in artifacts.items()) else None
    except Exception:
        return None
    finally:
        marker_path.unlink(missing_ok=True)


completed_before = (
    [video_id for video_id in ASSIGNED_VIDEO_IDS if valid_remote_completion(video_id)]
    if RUN_SCOPE == "SHARD"
    else []
)
pending = [video_id for video_id in ASSIGNED_VIDEO_IDS if video_id not in set(completed_before)]
if RUN_SCOPE == "PREFLIGHT":
    pending = []
print({
    "scope": RUN_SCOPE,
    "runner": RUNNER,
    "assigned_videos": len(ASSIGNED_VIDEO_IDS),
    "assigned_frames": len(ASSIGNED),
    "verified_before": len(completed_before),
    "pending_this_run": len(pending),
    "first_pending": pending[:5],
    "exact_plan_hash": FULL_PLAN_HASH,
})

## 4 — Decode mọi frame trong interval và embed SigLIP

In [ ]:
def literal_fps_for_video(video_id: str, target_dir: Path) -> float:
    mapping_path = target_dir / "mapping.csv"
    BTC.get(BTC_LAYOUT.map_keyframes(video_id), mapping_path)
    with mapping_path.open("r", encoding="utf-8-sig", newline="") as stream:
        rows = list(csv.DictReader(stream))
    values = {float(row["fps"]) for row in rows if row.get("fps")}
    if len(values) != 1:
        raise RuntimeError(f"{video_id}: mapping literal FPS không duy nhất: {values}")
    return next(iter(values))


def decode_exact_frames(
    video_path: Path,
    requests: list[dict],
    image_dir: Path,
    literal_fps: float,
) -> list[dict]:
    image_dir.mkdir(parents=True, exist_ok=True)
    requested = {int(row["frame_idx"]): row for row in requests}
    if len(requested) != len(requests):
        raise RuntimeError("Exact plan contains duplicate frame requests in one video")
    resolved = []
    with av.open(str(video_path)) as container:
        stream = container.streams.video[0]
        if stream.time_base is None:
            raise RuntimeError(f"{video_path}: decoder stream has no time_base")
        for frame_idx, frame in enumerate(container.decode(stream)):
            request = requested.get(frame_idx)
            if request is None:
                continue
            if frame.pts is None:
                raise RuntimeError(f"{request['keyframe_id']}: decoder frame has no PTS")
            target = image_dir / f"{frame_idx:09d}.jpg"
            temporary = target.with_suffix(".tmp.jpg")
            image = frame.to_image()
            try:
                image.save(temporary, format="JPEG", quality=JPEG_QUALITY)
            finally:
                image.close()
            temporary.replace(target)
            resolved.append({
                **request,
                "decoder_pts_time": float(frame.pts * stream.time_base),
                "literal_fps": literal_fps,
                "pts_time_source": "decoder_pts",
                "frame_idx_source": "decoder_counter",
                "jpeg_name": target.name,
                "jpeg_sha256": sha256_file(target),
            })
            if len(resolved) == len(requested):
                break
    found = {int(row["frame_idx"]) for row in resolved}
    missing = sorted(set(requested) - found)
    if missing:
        raise RuntimeError(f"{video_path}: decoder ended before exact frames {missing[:10]}")
    return sorted(resolved, key=lambda row: int(row["frame_idx"]))


def build_siglip(resolved: list[dict], image_dir: Path) -> tuple[list[str], np.ndarray]:
    ids = [str(row["keyframe_id"]) for row in resolved]
    chunks = []
    for start in range(0, len(resolved), SIGLIP_BATCH_SIZE):
        opened = []
        try:
            for row in resolved[start : start + SIGLIP_BATCH_SIZE]:
                with Image.open(image_dir / str(row["jpeg_name"])) as source:
                    opened.append(source.convert("RGB"))
            inputs = PROCESSOR(images=opened, return_tensors="pt").to("cuda")
            with torch.inference_mode():
                output = MODEL.get_image_features(**inputs)
                chunks.append(_siglip_feature_tensor(output).float().cpu().numpy())
        finally:
            for image in opened:
                image.close()
    matrix = np.concatenate(chunks, axis=0).astype(np.float32, copy=False)
    checked = check_vectors(ids, matrix, resolved[0]["video_id"], SIGLIP_DIMENSION)
    if checked.kept_ids != ids:
        raise RuntimeError("SigLIP validator dropped/reordered an exact sample")
    return ids, checked.kept

## 5 — Chạy shard; upload + verify rồi mới ghi marker

In [ ]:
started = time.monotonic()
failures = []
processed_this_session = 0
for ordinal, video_id in enumerate(pending, 1):
    if (time.monotonic() - started) / 60 >= SESSION_MAX_MINUTES:
        print("Đạt session time guard; Save Version lần sau sẽ resume.", flush=True)
        break
    local_dir = LOCAL / video_id
    image_dir = local_dir / "images"
    video_path = VIDEOS / f"{video_id}.mp4"
    local_dir.mkdir(parents=True, exist_ok=True)
    try:
        source_report_path = local_dir / "source-report.json"
        RESEARCH.get(f"{LB64_ROOT}/reports/{video_id}.json", source_report_path)
        source_report = json.loads(source_report_path.read_text(encoding="utf-8"))
        expected_video_sha = str(source_report["source_video_sha256"])
        literal_fps = literal_fps_for_video(video_id, local_dir)
        requests = BY_VIDEO[video_id]

        print(f"[{ordinal}/{len(pending)}] {video_id}: fetch video", flush=True)
        fetch_video(CATALOGUE[video_id], video_path)
        if sha256_file(video_path) != expected_video_sha:
            raise RuntimeError(f"{video_id}: source video SHA mismatch")
        print(f"[{ordinal}/{len(pending)}] {video_id}: decode {len(requests)} exact frame(s)", flush=True)
        resolved = decode_exact_frames(video_path, requests, image_dir, literal_fps)
        samples_path = write_jsonl_atomic(local_dir / "samples.jsonl", resolved)
        print(f"[{ordinal}/{len(pending)}] {video_id}: SigLIP", flush=True)
        ids, matrix = build_siglip(resolved, image_dir)
        siglip_path = local_dir / "siglip.npz"
        np.savez_compressed(siglip_path, ids=np.asarray(ids), vectors=matrix)
        images_path = write_deterministic_image_tar(
            {row["jpeg_name"]: image_dir / row["jpeg_name"] for row in resolved},
            local_dir / "images.tar",
        )

        report = {
            "schema_version": 1,
            "status": "O1S_EXACT_SIGLIP_VIDEO_COMPLETE",
            "decision_authority": "diagnostic_only",
            "run_id": RUN_ID,
            "runner": RUNNER,
            "video_id": video_id,
            "interval_policy": INTERVAL_POLICY,
            "full_interval_scan": True,
            "sample_count": len(resolved),
            "full_plan_hash": FULL_PLAN_HASH,
            "source_video_sha256": expected_video_sha,
            "source_report_sha256": sha256_file(source_report_path),
            "literal_fps": literal_fps,
            "frame_idx_source": "decoder_counter",
            "pts_time_source": "decoder_pts",
            "siglip": {
                "checkpoint": CHECKPOINT,
                "revision": MODEL_REVISION,
                "snapshot_hash": MODEL_SNAPSHOT_HASH,
                "processor": "SiglipImageProcessor(use_fast=True)",
                "dimension": SIGLIP_DIMENSION,
                "batch_size": SIGLIP_BATCH_SIZE,
                "canary_report_hash": CANARY_REPORT["report_hash"],
                "reference_lb64_canary_report_hash": REFERENCE_LB64_CANARY_REPORT_HASH,
            },
            "feature_parity_status": "MEASURED_SIGLIP_NUMERIC_CANARY_ONLY",
            "full_feature_parity_status": "BLOCKED_MISSING_OCR_CAPTION_BGE_AND_DOWNSTREAM",
            "heldout_accessed": False,
            "btc_mutated": False,
            "environment": freeze_environment(),
        }
        report["report_payload_hash"] = canonical_hash(report)
        report_path = write_json_atomic(local_dir / "report.json", report)
        remote = remote_paths(video_id)
        uploads = [
            (images_path, remote["images"]),
            (samples_path, remote["samples"]),
            (siglip_path, remote["siglip"]),
            (report_path, remote["report"]),
        ]
        RESEARCH.put_many(uploads)
        artifact_hashes = {remote_path: sha256_file(local_path) for local_path, remote_path in uploads}
        for remote_path, expected_hash in artifact_hashes.items():
            if RESEARCH.sha256(remote_path) != expected_hash:
                raise RuntimeError(f"{video_id}: remote hash mismatch: {remote_path}")

        marker = {
            "schema_version": 1,
            "status": "COMPLETE",
            "run_id": RUN_ID,
            "runner": RUNNER,
            "video_id": video_id,
            "full_plan_hash": FULL_PLAN_HASH,
            "interval_policy": INTERVAL_POLICY,
            "sample_count": len(resolved),
            "artifacts": artifact_hashes,
            "heldout_accessed": False,
            "btc_mutated": False,
        }
        marker["marker_payload_hash"] = canonical_hash(marker)
        marker_path = write_json_atomic(local_dir / "COMPLETE.json", marker)
        RESEARCH.put(marker_path, remote["marker"])
        if RESEARCH.sha256(remote["marker"]) != sha256_file(marker_path):
            raise RuntimeError(f"{video_id}: remote marker SHA mismatch")
        if valid_remote_completion(video_id) is None:
            raise RuntimeError(f"{video_id}: completion marker contract validation failed")
        processed_this_session += 1
        print(f"[{ordinal}/{len(pending)}] {video_id}: COMPLETE", flush=True)
    except Exception as error:
        failures.append({"video_id": video_id, "error_type": type(error).__name__, "error": str(error)})
        print({"failed": failures[-1]}, flush=True)
    finally:
        video_path.unlink(missing_ok=True)
        if local_dir.exists():
            shutil.rmtree(local_dir, ignore_errors=True)
        gc.collect()
        torch.cuda.empty_cache()

## 6 — Worker manifest và trạng thái bàn giao

In [ ]:
verified = [video_id for video_id in ASSIGNED_VIDEO_IDS if valid_remote_completion(video_id)] if RUN_SCOPE == "SHARD" else []
status = (
    "PREFLIGHT_PASS" if RUN_SCOPE == "PREFLIGHT"
    else "COMPLETE" if len(verified) == len(ASSIGNED_VIDEO_IDS)
    else "PARTIAL"
)
worker_manifest = {
    "schema_version": 1,
    "status": status,
    "decision_authority": "diagnostic_only",
    "run_id": RUN_ID,
    "runner": RUNNER,
    "runners": RUNNERS,
    "interval_policy": INTERVAL_POLICY,
    "full_plan_hash": FULL_PLAN_HASH,
    "full_interval_targets": len(FULL_TARGETS),
    "exact_unique_frames": len(FULL_PLAN),
    "notebook18_report_sha256": EXPECTED_NB18_REPORT_SHA256,
    "targets_sha256": EXPECTED_TARGETS_SHA256,
    "notebook20_complete_sha256": EXPECTED_AGGREGATE_COMPLETE_SHA256,
    "assigned": ASSIGNED_VIDEO_IDS,
    "assigned_count": len(ASSIGNED_VIDEO_IDS),
    "assigned_frame_count": len(ASSIGNED),
    "verified_complete": verified,
    "verified_complete_count": len(verified),
    "processed_this_session": processed_this_session,
    "failures_this_session": failures,
    "repo_commit": REPO_COMMIT,
    "siglip_canary_report_hash": CANARY_REPORT["report_hash"],
    "heldout_accessed": False,
    "btc_mutated": False,
}
worker_manifest["manifest_payload_hash"] = canonical_hash(worker_manifest)
manifest_path = write_json_atomic(WORK_DIR / f"{RUNNER}-manifest.json", worker_manifest)
manifest_remote = None
if RUN_SCOPE == "SHARD":
    manifest_remote = f"{OUTPUT_ROOT}/worker-manifests/{RUNNER}.json"
    RESEARCH.put(manifest_path, manifest_remote)
    if RESEARCH.sha256(manifest_remote) != sha256_file(manifest_path):
        raise RuntimeError("Worker manifest remote SHA mismatch")
print(json.dumps({
    "status": status,
    "runner": RUNNER,
    "assigned": len(ASSIGNED_VIDEO_IDS),
    "assigned_frames": len(ASSIGNED),
    "verified": len(verified),
    "processed_this_session": processed_this_session,
    "failures_this_session": failures,
    "manifest": manifest_remote,
    "next": "Hai runner COMPLETE -> notebook 22 exact O1-S aggregate",
}, ensure_ascii=False, indent=2))

## Cách đọc kết quả

- `PREFLIGHT_PASS`: repo, dependency, lineage, exact plan và numeric canary đều đúng; chưa decode video.
- `PARTIAL`: runner đã xử lý một phần; Save Version lại đúng runner để resume các video còn thiếu.
- `COMPLETE`: toàn bộ video của runner đã có đủ 4 artifact và marker hash hợp lệ.
- Hai runner `COMPLETE` là điều kiện để chạy notebook 22 aggregate exact O1-S.
- Notebook 21 vẫn chỉ đo dense SigLIP. Full OCR/caption/BGE/downstream feature parity và
  official score vẫn phải qua các gate tiếp theo.